In [1]:
"""
Example: Walk-Forward Portfolio Optimization.

Demonstrates the rolling-window strategy on synthetic 27-year data
with IPOs and delistings. Replace the synthetic data section with
your own DataFrame.
"""
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# sys.path.insert(0, str(Path(__file__).parent.parent))

import os
os.chdir("..")

# from rl_agent.config import (
#    Config, 
#    EnvironmentConfig, FeatureConfig, 
#    NetworkConfig, TrainingConfig, BacktestConfig
# )
from rl_agent.fin_data import download_fin_data, get_sp500
from rl_agent.forwardbacktest import WalkForwardBacktestEngine
from rl_agent.utils import (
    compute_portfolio_metrics,
    format_metrics,
    generate_realistic_universe
)

ticker, sp500 = get_sp500()
prices = pd.read_csv("C:\\Users\\lukas\\Downloads\\prices.csv", index_col=0, parse_dates=True)
engine = WalkForwardBacktestEngine(prices)


date       str
tickers    str
dtype: object
1194


In [2]:
results = engine.run(kfold_init=True)

Reinforcement Learning Portfolio Optimization
Dynamic Universe Summary:
  Total unique stocks:        818
  Max concurrent stocks:      500
  Min concurrent stocks:      188
  Mean concurrent stocks:     338
  Entrants:                   474
  Delistings:                 162
  Date range:                 1996-01-02 to 2026-01-30
  Data size:                  818
  Train window:     5.0 years
  Warm-start:       True
  Episodes/window:  500
  K-Fold Cross-Validation Initialization
  Data:   1996-01-02 → 2000-07-31 (1157 days)
  Splits: 5  |  Post-warmup dates: 653  |  Warmup: 504 days

  Fold 1/5: train 1996-01-02 → 2000-07-31, val 1997-12-30 → 2000-07-28 (131 sampled dates, 651d range)
  RL Portfolio Optimization - Training
  Device:   cuda
  Episodes: 1000
Episode    1/1000 | Reward:  -0.4032 | Avg (last 50):  -0.4032 | Policy loss: -0.8782 | Value loss: 3.3704 | Time: 2s
Episode   10/1000 | Reward:  -0.0988 | Avg (last 50):  -0.3007 | Policy loss: -0.8513 | Value loss: 1.7800 | Time:

In [3]:
# Analyze results
print("\n" + "=" * 50)
print("  Detailed Results")
print("=" * 50)

# Annual return breakdown
oos = results["oos_series"]
if len(oos) > 12:
    annual = (1 + oos).groupby(oos.index.year).prod() - 1
    print(f"\n  Annual OOS returns:")
    for year, ret in annual.items():
        print(f"    {year}: {ret:+.2%}")

#  Overall metrics
# rl_returns = results["oos_returns"]
# rl_values = np.cumprod(np.concatenate([[1.0], 1 + rl_returns]))
# pd.DataFrame(rl_values).to_csv("C:\\Users\\lukas\\Downloads\\rl_portfolio_values.csv")
# rl_metrics = compute_portfolio_metrics(rl_values, rl_returns, periods_per_year=12)
# print(format_metrics(rl_metrics))

# Weight concentration
weight_df = engine.get_weight_history()
if len(weight_df) > 0:
    top5_avg = weight_df.apply(
        lambda row: row.nlargest(5).sum(), axis=1
    ).mean()
    n_nonzero_avg = (weight_df.abs() > 0.001).sum(axis=1).mean()
    print(f"\n  Weight statistics:")
    print(f"    Average top-5 concentration: {top5_avg:.2%}")
    print(f"    Average non-zero positions:  {n_nonzero_avg:.0f}")

weight_df.to_csv("C:\\Users\\lukas\\Downloads\\weights_rl.csv")

# Compare to equal-weight benchmark. Use same daily return data as RL (already winsorized once in the engine)
returns = engine.returns.fillna(0)

# Active mask
period_returns = []
active_mask = pd.DataFrame()  
for r in results["window_results"]:
    active_mask = pd.concat([active_mask, pd.DataFrame([r.active_mask])])
    period_df = returns.loc[r.oos_start:r.oos_end]
    period_returns.append((1 + period_df).prod() - 1)
pd.DataFrame(active_mask).to_csv("C:\\Users\\lukas\\Downloads\\active_mask.csv")

# bm_prices = engine.prices.mask(engine.prices < 1)
# returns = bm_prices.pct_change()
# returns = returns.apply(lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99)), axis=1).fillna(0)

#  Compute RL and BM portfolio returns by applying the same active mask to the returns, then averaging across active stocks
# masked_returns = pd.DataFrame(period_returns).where(active_mask.values, np.nan)
# bm_returns = masked_returns.mean(axis=1).fillna(0).values
# portfolio_returns = pd.DataFrame(masked_returns.values * weight_df.values)  
# pt_returns = portfolio_returns.sum(axis=1).fillna(0).values
# pt_values = np.cumprod(np.concatenate([[1.0], 1 + pt_returns]))
# pd.DataFrame(pt_values).to_csv("C:\\Users\\lukas\\Downloads\\pt_portfolio_values.csv")

# pt_metrics = compute_portfolio_metrics(pt_values, pt_returns, periods_per_year=12)
# print(format_metrics(pt_metrics))

bm_period_returns = []
for r in results["window_results"]:
    active = r.active_mask[:prices.shape[1]]
    n_active = int(active.sum())
    ew = np.zeros(prices.shape[1])
    if n_active > 0:
        ew[active] = 1.0 / n_active
    period_df = returns.loc[r.oos_start:r.oos_end]
    daily_ew = period_df.values @ ew
    bm_period_returns.append(float(np.prod(1 + daily_ew) - 1))
bm_returns = np.array(bm_period_returns)

# Compute equal-weight TC: each period weights drift with returns,
# and must be rebalanced back to equal weight at the next period start.
tc_rate_bm = (engine.env_config.transaction_cost_bps + engine.env_config.slippage_bps) / 10_000
bm_tc_per_period = []
prev_ew_end = None
for r in results["window_results"]:
    active = r.active_mask[:prices.shape[1]]
    n_active = int(active.sum())
    if n_active == 0:
        bm_tc_per_period.append(0.0)
        continue
    ew = np.zeros(prices.shape[1])
    ew[active] = 1.0 / n_active
    turnover = np.sum(np.abs(ew - prev_ew_end)) if prev_ew_end is not None else ew.sum()
    bm_tc_per_period.append(turnover * tc_rate_bm)
    cum = (1 + returns.loc[r.oos_start:r.oos_end]).prod().values[:len(ew)]
    end_w = ew * cum
    s = end_w.sum()
    prev_ew_end = end_w / s if s > 1e-10 else ew.copy()
bm_net_rets = bm_returns - np.array(bm_tc_per_period)

bm_values = np.cumprod(np.concatenate([[1.0], 1 + bm_returns]))
pd.DataFrame(bm_values).to_csv("C:\\Users\\lukas\\Downloads\\bm_portfolio_values.csv")
    
bm_metrics = compute_portfolio_metrics(bm_values, bm_returns, periods_per_year=12)
print(format_metrics(bm_metrics))

# Generate HTML dashboard

# One date per OOS period (monthly data → one date per bar/point)
oos_dates = pd.DatetimeIndex([r.oos_start for r in results["window_results"]])

dashboard_kwargs = dict(
    rl_results=results["oos_returns"],
    bm_daily_returns=bm_returns,
    rl_dates=oos_dates,
    weight_history=weight_df if len(weight_df) > 0 else None,
    output_path="rl_backtest_dashboard.html",
    title="RL Portfolio vs Equal-Weight Benchmark",
    periods_per_year=12,
    rl_gross_returns=results["oos_raw_returns"],
    bm_net_returns=bm_net_rets,
)

# Prefer Plotly (utils.py); fall back to Chart.js (utils_html.py)
try:
    from rl_agent.utils import generate_dashboard
    generate_dashboard(**dashboard_kwargs)
except ImportError:
    from rl_agent.utils_html import generate_dashboard
    print("  (Plotly not installed, using Chart.js fallback)")
    generate_dashboard(**dashboard_kwargs)


  Detailed Results

  Annual OOS returns:
    2000: +15.33%
    2001: +3.53%
    2002: -11.93%
    2003: +39.07%
    2004: +20.35%
    2005: +8.74%
    2006: +15.92%
    2007: +2.61%
    2008: -35.10%
    2009: +40.77%
    2010: +20.05%
    2011: +0.18%
    2012: +17.24%
    2013: +34.45%
    2014: +13.20%
    2015: -3.46%
    2016: +14.75%
    2017: +19.99%
    2018: -7.46%
    2019: +27.22%
    2020: +7.12%
    2021: +29.40%
    2022: -10.36%
    2023: -2.35%
    2024: +34.26%
    2025: +7.03%
    2026: +3.65%

  Weight statistics:
    Average top-5 concentration: 3.94%
    Average non-zero positions:  347
  Total Return                 1619.49%
  Annualized Return            12.58%
  Annualized Volatility        17.31%
  Sharpe Ratio                 0.776
  Sortino Ratio                1.005
  Max Drawdown                 51.54%
  Max DD Duration (days)       42
  Calmar Ratio                 0.244
  VaR (95%)                    -0.0688
  CVaR (95%)                   -0.1093
  Dash